In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

### Một số thư viện cần thiết

In [ ]:
behavior = pd.read_excel(r"C:\Users\DELL\Downloads\widsdatathon2025\TRAIN_NEW\TRAIN_QUANTITATIVE_METADATA_new.xlsx")
demographics = pd.read_excel(r"C:\Users\DELL\Downloads\widsdatathon2025\TRAIN_NEW\TRAIN_CATEGORICAL_METADATA_new.xlsx")
fmri = pd.read_csv(r"C:\Users\DELL\Downloads\widsdatathon2025\TRAIN_NEW\TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv")
labels = pd.read_excel(r"C:\Users\DELL\Downloads\widsdatathon2025\TRAIN_NEW\TRAINING_SOLUTIONS.xlsx")

df = behavior.merge(demographics, on="participant_id", how="inner") \
             .merge(fmri, on="participant_id", how="inner") \
             .merge(labels, on="participant_id", how="inner")

### Mục đích: Gộp 4 dataframe (behavior, demographics, fmri, labels) lại thành một bảng duy nhất df dựa vào cột chung participant_id.
### how="inner": chỉ giữ lại những participant_id có mặt trong tất cả các bảng.

In [ ]:
df_sample = df.sample(frac=1, random_state=42)

print(f"Tổng số dòng trong df: {len(df)}")
print(f"Số dòng sau khi sample 10%: {len(df_sample)}")

### sample(frac=1) nghĩa là xáo trộn toàn bộ dòng dữ liệu (100%)

In [ ]:
X = df_sample.drop(columns=["participant_id", "ADHD_Outcome", "Sex_F"])
y_adhd = df_sample["ADHD_Outcome"]
y_sex = df_sample["Sex_F"]

### X: biến đầu vào dùng để huấn luyện, bỏ participant_id, ADHD_Outcome, Sex_F vì đây là thông tin nhãn hoặc ID.

### y_adhd, y_sex: biến mục tiêu (labels) để huấn luyện mô hình phân loại ADHD và giới tính.

In [ ]:
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

### Bước 1: xử lý giá trị thiếu (NaN) trong X bằng cách thay bằng giá trị trung bình theo cột.

### Bước 2: chuẩn hóa dữ liệu đầu vào về trung bình 0, độ lệch chuẩn 1 để mô hình học tốt hơn (đặc biệt cần khi dùng SVM, KNN, logistic…).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_adhd, test_size=0.2, random_state=42)

print("\nADHD Test Label Distribution:", np.bincount(y_test))

### Chia tập train/test theo tỷ lệ 80% - 20%.

### In ra số mẫu thuộc mỗi lớp (0 hoặc 1) trong tập test → giúp đánh giá mất cân bằng.

In [ ]:
clf_adhd = RandomForestClassifier(n_estimators=100, random_state=42,class_weight="balanced")
clf_adhd.fit(X_train, y_train)

### Tạo và huấn luyện một mô hình Random Forest:

### - n_estimators=100: dùng 100 cây quyết định.

### - class_weight="balanced": tự động điều chỉnh trọng số lớp → khắc phục mất cân bằng nhãn.

In [ ]:
y_pred_adhd = clf_adhd.predict(X_test)

### Dự đoán kết quả ADHD từ tập test.

In [ ]:
print("==== ADHD REPORT ====")
print(classification_report(y_test, y_pred_adhd))

### In ra các chỉ số đánh giá: precision, recall, f1-score, support theo từng lớp.

In [ ]:
print("\nSố mẫu ADHD:")
print(df_sample["ADHD_Outcome"].value_counts())

### In số lượng mỗi lớp trong toàn bộ dữ liệu → kiểm tra mức độ mất cân bằng nhãn.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_sex, test_size=0.2, random_state=42)

print("\nSex Test Label Distribution:", np.bincount(y_test))

In [ ]:
clf_sex = RandomForestClassifier(n_estimators=100, random_state=42,class_weight="balanced")
clf_sex.fit(X_train, y_train)

In [ ]:
y_pred_sex = clf_sex.predict(X_test)

In [ ]:
print("==== SEX REPORT ====")
print(classification_report(y_test, y_pred_sex))

In [ ]:
print("\nSố mẫu SEX:")
print(df_sample["Sex_F"].value_counts())

In [ ]:
behavior_test = pd.read_excel(r"C:\Users\DELL\Downloads\widsdatathon2025\TEST\TEST_QUANTITATIVE_METADATA.xlsx")
demographics_test = pd.read_excel(r"C:\Users\DELL\Downloads\widsdatathon2025\TEST\TEST_CATEGORICAL.xlsx")
fmri_test = pd.read_csv(r"C:\Users\DELL\Downloads\widsdatathon2025\TEST\TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv")

df_test = behavior_test.merge(demographics_test, on="participant_id", how="inner") \
                       .merge(fmri_test, on="participant_id", how="inner")

### Gộp dữ liệu submission từ 3 file

In [ ]:
X_test_real = df_test.drop(columns=["participant_id"])

### Bỏ cột participant_id

In [ ]:
X_test_imputed = imputer.transform(X_test_real)  
X_test_scaled = scaler.transform(X_test_imputed)

### Tiền xử lý như dữ liệu train

In [ ]:
y_pred_adhd_test = clf_adhd.predict(X_test_scaled)
y_pred_sex_test = clf_sex.predict(X_test_scaled)

### Dự đoán từ mô hình đã train 

In [ ]:
df_submit = pd.DataFrame({
    "participant_id": df_test["participant_id"],
    "ADHD_Outcome": y_pred_adhd_test,
    "Sex_F": y_pred_sex_test
})

df_submit.to_csv(r"C:\Users\DELL\Downloads\widsdatathon2025\submission.csv", index=False)
print("Đã lưu kết quả dự đoán vào submission.csv!")

### Lưu file kết quả    